# Anchor Top

# Load libraries

In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Configure Polars to show all rows and make it scrollable
pl.Config.set_tbl_rows(-1)  # Show all rows
pl.Config.set_tbl_cols(-1)  # Show all columns
pl.Config.set_tbl_width_chars(1000)  # Set wider table width

polars.config.Config

# Load Data

In [8]:
# Load all data files using Polars
print("Loading datasets with Polars...")

# Main application data
train_df = pl.read_csv('../data/application_train.csv')
test_df = pl.read_csv('../data/application_test.csv')

# Additional data sources
bureau_df = pl.read_csv('../data/bureau.csv')
bureau_balance_df = pl.read_csv('../data/bureau_balance.csv')
previous_app_df = pl.read_csv('../data/previous_application.csv')
credit_card_df = pl.read_csv('../data/credit_card_balance.csv')
pos_cash_df = pl.read_csv('../data/POS_CASH_balance.csv')
installments_df = pl.read_csv('../data/installments_payments.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"Bureau data shape: {bureau_df.shape}")
print(f"Previous applications shape: {previous_app_df.shape}")
print(f"Credit card balance shape: {credit_card_df.shape}")
print(f"POS Cash balance shape: {pos_cash_df.shape}")
print(f"Installments payments shape: {installments_df.shape}")

print(f"\nTarget distribution:")
target_counts = train_df.select(pl.col('TARGET')).to_pandas()['TARGET'].value_counts(normalize=True)
print(target_counts)

Loading datasets with Polars...
Training data shape: (307511, 122)
Test data shape: (48744, 121)
Bureau data shape: (1716428, 17)
Previous applications shape: (1670214, 37)
Credit card balance shape: (3840312, 23)
POS Cash balance shape: (10001358, 8)
Installments payments shape: (13605401, 8)

Target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


# Define Utils Function

In [11]:
def create_customer_segment_features_fixed(train_df, bureau_df, previous_app_df):
    """
    Create customer segmentation features (FIXED VERSION):
    - First-time Home Credit customers
    - Credit virgins (no external credit history)  
    - Previous application patterns
    """
    
    print("=== CREATING CUSTOMER SEGMENT FEATURES ===")
    
    # Start with main training data
    enhanced_df = train_df.clone()
    
    # 1. HOME CREDIT EXPERIENCE FEATURES
    print("\n1. Analyzing Home Credit customer history...")
    
    # Count previous Home Credit applications per customer
    prev_app_stats = previous_app_df.group_by('SK_ID_CURR').agg([
        pl.count().alias('PREV_APP_COUNT'),
        pl.col('NAME_CONTRACT_STATUS').filter(pl.col('NAME_CONTRACT_STATUS') == 'Approved').count().alias('PREV_APP_APPROVED'),
        pl.col('NAME_CONTRACT_STATUS').filter(pl.col('NAME_CONTRACT_STATUS') == 'Refused').count().alias('PREV_APP_REFUSED'),
        pl.col('AMT_APPLICATION').mean().alias('PREV_APP_AMT_MEAN'),
        pl.col('AMT_CREDIT').mean().alias('PREV_APP_CREDIT_MEAN')
    ])
    
    # Join with main data
    enhanced_df = enhanced_df.join(prev_app_stats, on='SK_ID_CURR', how='left')
    
    # Fill nulls (customers with no previous applications)
    enhanced_df = enhanced_df.with_columns([
        pl.col('PREV_APP_COUNT').fill_null(0),
        pl.col('PREV_APP_APPROVED').fill_null(0), 
        pl.col('PREV_APP_REFUSED').fill_null(0),
        pl.col('PREV_APP_AMT_MEAN').fill_null(0),
        pl.col('PREV_APP_CREDIT_MEAN').fill_null(0)
    ])
    
    # 2. EXTERNAL CREDIT BUREAU FEATURES
    print("2. Analyzing external credit bureau history...")
    
    # Count external credit relationships per customer
    bureau_stats = bureau_df.group_by('SK_ID_CURR').agg([
        pl.count().alias('BUREAU_COUNT'),
        pl.col('CREDIT_ACTIVE').filter(pl.col('CREDIT_ACTIVE') == 'Active').count().alias('BUREAU_ACTIVE_COUNT'),
        pl.col('CREDIT_ACTIVE').filter(pl.col('CREDIT_ACTIVE') == 'Closed').count().alias('BUREAU_CLOSED_COUNT'),
        pl.col('DAYS_CREDIT').min().alias('DAYS_FIRST_CREDIT'),  # Oldest credit
        pl.col('AMT_CREDIT_SUM').sum().alias('BUREAU_TOTAL_DEBT'),
        pl.col('AMT_CREDIT_SUM_OVERDUE').sum().alias('BUREAU_TOTAL_OVERDUE')
    ])
    
    # Join with main data
    enhanced_df = enhanced_df.join(bureau_stats, on='SK_ID_CURR', how='left')
    
    # Fill nulls (customers with no bureau history)
    enhanced_df = enhanced_df.with_columns([
        pl.col('BUREAU_COUNT').fill_null(0),
        pl.col('BUREAU_ACTIVE_COUNT').fill_null(0),
        pl.col('BUREAU_CLOSED_COUNT').fill_null(0),
        pl.col('DAYS_FIRST_CREDIT').fill_null(0),
        pl.col('BUREAU_TOTAL_DEBT').fill_null(0),
        pl.col('BUREAU_TOTAL_OVERDUE').fill_null(0)
    ])
    
    # 3. CREATE CUSTOMER SEGMENT FLAGS
    print("3. Creating customer segment flags...")
    
    enhanced_df = enhanced_df.with_columns([
        # First-time Home Credit customer
        (pl.col('PREV_APP_COUNT') == 0).cast(pl.Int32).alias('IS_FIRST_HOME_CREDIT_CUSTOMER'),
        
        # Credit virgin (no external credit history)
        (pl.col('BUREAU_COUNT') == 0).cast(pl.Int32).alias('IS_CREDIT_VIRGIN'),
        
        # Home Credit approval rate (for non-first-time customers)
        pl.when(pl.col('PREV_APP_COUNT') > 0)
        .then(pl.col('PREV_APP_APPROVED') / pl.col('PREV_APP_COUNT'))
        .otherwise(0).alias('HOME_CREDIT_APPROVAL_RATE'),
        
        # External credit experience in months (convert days to months)
        pl.when(pl.col('DAYS_FIRST_CREDIT') < 0)
        .then((pl.col('DAYS_FIRST_CREDIT') * -1) / 30.44)  # Convert to months
        .otherwise(0).alias('MONTHS_CREDIT_HISTORY'),
        
        # Debt-to-income ratio from external sources
        pl.when(pl.col('AMT_INCOME_TOTAL') > 0)
        .then(pl.col('BUREAU_TOTAL_DEBT') / pl.col('AMT_INCOME_TOTAL'))
        .otherwise(0).alias('EXTERNAL_DEBT_INCOME_RATIO'),
    ])
    
    # 4. CUSTOMER RISK SEGMENTS (separate step using pl.lit for string literals)
    print("4. Creating customer risk segments...")
    
    enhanced_df = enhanced_df.with_columns([
        # Risk segment based on credit experience
        pl.when((pl.col('IS_CREDIT_VIRGIN') == 1) & (pl.col('IS_FIRST_HOME_CREDIT_CUSTOMER') == 1))
        .then(pl.lit('HIGH_RISK_NEW_TO_CREDIT'))
        .when((pl.col('IS_CREDIT_VIRGIN') == 0) & (pl.col('IS_FIRST_HOME_CREDIT_CUSTOMER') == 1))
        .then(pl.lit('MEDIUM_RISK_NEW_TO_HOME_CREDIT')) 
        .when((pl.col('IS_CREDIT_VIRGIN') == 0) & (pl.col('IS_FIRST_HOME_CREDIT_CUSTOMER') == 0) & (pl.col('HOME_CREDIT_APPROVAL_RATE') > 0.5))
        .then(pl.lit('LOW_RISK_REPEAT_APPROVED'))
        .when((pl.col('IS_CREDIT_VIRGIN') == 0) & (pl.col('IS_FIRST_HOME_CREDIT_CUSTOMER') == 0) & (pl.col('HOME_CREDIT_APPROVAL_RATE') <= 0.5))
        .then(pl.lit('MEDIUM_RISK_REPEAT_REJECTED'))
        .otherwise(pl.lit('UNKNOWN_RISK')).alias('CUSTOMER_RISK_SEGMENT')
    ])
    
    return enhanced_df

In [13]:
# Let's start simple and build step by step
print("=== SIMPLE CUSTOMER SEGMENTATION ===")

# # Load data
# train_df = pl.read_csv('data/application_train.csv')
# bureau_df = pl.read_csv('data/bureau.csv') 
# previous_app_df = pl.read_csv('data/previous_application.csv')

print(f"Data loaded successfully!")
print(f"Train: {train_df.shape}")
print(f"Bureau: {bureau_df.shape}")
print(f"Previous apps: {previous_app_df.shape}")

# Step 1: Count previous Home Credit applications
print("\n1. Counting previous Home Credit applications...")
prev_counts = previous_app_df.group_by('SK_ID_CURR').agg([
    pl.count().alias('PREV_APP_COUNT')
])
print(f"Previous app counts: {prev_counts.shape}")

# Step 2: Count bureau records  
print("\n2. Counting bureau records...")
bureau_counts = bureau_df.group_by('SK_ID_CURR').agg([
    pl.count().alias('BUREAU_COUNT')
])
print(f"Bureau counts: {bureau_counts.shape}")

# Step 3: Join with main data
print("\n3. Joining with main dataset...")
enhanced = train_df.select(['SK_ID_CURR', 'TARGET']).join(prev_counts, on='SK_ID_CURR', how='left')
enhanced = enhanced.join(bureau_counts, on='SK_ID_CURR', how='left')

# Fill nulls
enhanced = enhanced.with_columns([
    pl.col('PREV_APP_COUNT').fill_null(0),
    pl.col('BUREAU_COUNT').fill_null(0)
])

print(f"Enhanced dataset: {enhanced.shape}")
print(enhanced.head())

=== SIMPLE CUSTOMER SEGMENTATION ===
Data loaded successfully!
Train: (307511, 122)
Bureau: (1716428, 17)
Previous apps: (1670214, 37)

1. Counting previous Home Credit applications...
Previous app counts: (338857, 2)

2. Counting bureau records...
Bureau counts: (305811, 2)

3. Joining with main dataset...
Enhanced dataset: (307511, 4)
shape: (5, 4)
┌────────────┬────────┬────────────────┬──────────────┐
│ SK_ID_CURR ┆ TARGET ┆ PREV_APP_COUNT ┆ BUREAU_COUNT │
│ ---        ┆ ---    ┆ ---            ┆ ---          │
│ i64        ┆ i64    ┆ u32            ┆ u32          │
╞════════════╪════════╪════════════════╪══════════════╡
│ 100002     ┆ 1      ┆ 1              ┆ 8            │
│ 100003     ┆ 0      ┆ 3              ┆ 4            │
│ 100004     ┆ 0      ┆ 1              ┆ 2            │
│ 100006     ┆ 0      ┆ 9              ┆ 0            │
│ 100007     ┆ 0      ┆ 6              ┆ 1            │
└────────────┴────────┴────────────────┴──────────────┘


In [15]:
# Step 4: Create simple binary flags
print("\n4. Creating binary flags...")

enhanced = enhanced.with_columns([
    # First-time Home Credit customer (no previous applications)
    (pl.col('PREV_APP_COUNT') == 0).cast(pl.Int8).alias('IS_FIRST_HOME_CREDIT_CUSTOMER'),
    
    # Credit virgin (no bureau records)
    (pl.col('BUREAU_COUNT') == 0).cast(pl.Int8).alias('IS_CREDIT_VIRGIN')
])

print("Binary flags created successfully!")
print(enhanced.head())

# Step 5: Analyze the segments
print("\n5. Analyzing customer segments...")

print("First-time Home Credit customers:")
first_time_stats = enhanced.group_by('IS_FIRST_HOME_CREDIT_CUSTOMER').agg([
    pl.count().alias('count'),
    pl.col('TARGET').mean().alias('default_rate')
])
print(first_time_stats)

print("\nCredit virgins:")
credit_virgin_stats = enhanced.group_by('IS_CREDIT_VIRGIN').agg([
    pl.count().alias('count'),
    pl.col('TARGET').mean().alias('default_rate')
])
print(credit_virgin_stats)

# Combined analysis
print("\nCombined analysis:")
combined_stats = enhanced.group_by(['IS_FIRST_HOME_CREDIT_CUSTOMER', 'IS_CREDIT_VIRGIN']).agg([
    pl.count().alias('count'),
    pl.col('TARGET').mean().alias('default_rate')
]).sort('default_rate', descending=True)
print(combined_stats)


4. Creating binary flags...
Binary flags created successfully!
shape: (5, 6)
┌────────────┬────────┬────────────────┬──────────────┬───────────────────────────────┬──────────────────┐
│ SK_ID_CURR ┆ TARGET ┆ PREV_APP_COUNT ┆ BUREAU_COUNT ┆ IS_FIRST_HOME_CREDIT_CUSTOMER ┆ IS_CREDIT_VIRGIN │
│ ---        ┆ ---    ┆ ---            ┆ ---          ┆ ---                           ┆ ---              │
│ i64        ┆ i64    ┆ u32            ┆ u32          ┆ i8                            ┆ i8               │
╞════════════╪════════╪════════════════╪══════════════╪═══════════════════════════════╪══════════════════╡
│ 100002     ┆ 1      ┆ 1              ┆ 8            ┆ 0                             ┆ 0                │
│ 100003     ┆ 0      ┆ 3              ┆ 4            ┆ 0                             ┆ 0                │
│ 100004     ┆ 0      ┆ 1              ┆ 2            ┆ 0                             ┆ 0                │
│ 100006     ┆ 0      ┆ 9              ┆ 0            ┆ 0         

# Simple EDA